# Online vs. Offline Learning — Empirical Analysis

---

## Dataset Description

| Property | Detail |
|----------|--------|
| **Source** | `online_vs_offline_learning_dataset.csv` |
| **Rows × Columns** | 1,000 × 6 |
| **Balance** | 500 Online, 500 Offline |
| **Subjects** | English, History, Math, Programming, Science |
| **Variables** | `Learning_Mode`, `Subject`, `Study_Hours`, `Retention_Score`, `Focus_Level`, `Exam_Score` |

---

## Research Questions

1. **Performance gap**: Does learning mode (Online vs. Offline) significantly affect exam performance, retention score, and focus level?
2. **Predictors**: What are the strongest predictors of Exam Score, and does learning mode moderate these relationships?
3. **Subject-level differences**: Do Online and Offline learners differ in subject-specific exam performance?

---


In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# ── Style ────────────────────────────────────────────────────────────────────
try:
    plt.style.use('seaborn-v0_8')
except OSError:
    try:
        plt.style.use('seaborn')
    except OSError:
        plt.style.use('ggplot')

PALETTE    = {'Online': '#2196F3', 'Offline': '#FF9800'}   # blue / orange
FONT_TITLE = dict(fontsize=14, fontweight='bold')
FONT_AXIS  = dict(fontsize=11)

# ── Load data ────────────────────────────────────────────────────────────────
DATA_PATH = '../online_vs_offline_learning_dataset.csv'
df = pd.read_csv(DATA_PATH)

print(f'Shape : {df.shape}')
print(f'Columns : {list(df.columns)}')
print(f"\nLearning_Mode counts:\n{df['Learning_Mode'].value_counts().to_string()}")
print(f"\nSubject counts:\n{df['Subject'].value_counts().to_string()}")
print()
df.head()

## 1. Descriptive Statistics

We first compute mean ± standard deviation for each numeric variable, grouped by `Learning_Mode`.


In [ ]:
NUMERIC = ['Exam_Score', 'Retention_Score', 'Focus_Level', 'Study_Hours']

online  = df[df['Learning_Mode'] == 'Online']
offline = df[df['Learning_Mode'] == 'Offline']

# Compute mean and std per group
summary_mean = df.groupby('Learning_Mode')[NUMERIC].mean().round(2)
summary_std  = df.groupby('Learning_Mode')[NUMERIC].std().round(2)

# Format as "mean ± std" strings
formatted = pd.DataFrame(index=summary_mean.index)
for col in NUMERIC:
    formatted[col] = (summary_mean[col].astype(str)
                      + ' ± '
                      + summary_std[col].astype(str))

formatted.index.name = 'Learning_Mode'
print('Descriptive Statistics — Mean ± Std by Learning Mode')
print('=' * 60)
formatted

## 2. Hypothesis Testing: Online vs. Offline

For each numeric variable we run:
- **Independent samples t-test** (assumes approximate normality; n=500 per group, so CLT applies)
- **Mann-Whitney U test** (non-parametric, no distributional assumption)

Significance thresholds: `***` p<0.001 · `**` p<0.01 · `*` p<0.05 · `ns` not significant


In [ ]:
rows = []
for col in NUMERIC:
    a = online[col].dropna()
    b = offline[col].dropna()
    t_stat, t_p = stats.ttest_ind(a, b)
    u_stat, u_p = stats.mannwhitneyu(a, b, alternative='two-sided')
    sig = '***' if t_p < 0.001 else ('**' if t_p < 0.01 else ('*' if t_p < 0.05 else 'ns'))
    rows.append({
        'Metric':         col,
        'Online Mean':    round(a.mean(), 2),
        'Offline Mean':   round(b.mean(), 2),
        'Diff (Off-On)':  round(b.mean() - a.mean(), 2),
        't-statistic':    round(t_stat, 3),
        't p-value':      round(t_p, 4),
        'U-statistic':    int(u_stat),
        'MWU p-value':    round(u_p, 4),
        'Significance':   sig
    })

results_df = pd.DataFrame(rows).set_index('Metric')
print('Hypothesis Tests — Online vs. Offline Learning')
print('=' * 70)
results_df

### Interpretation of Hypothesis Tests

| Metric | Result | Interpretation |
|--------|--------|----------------|
| **Exam_Score** | Offline=73.14, Online=70.21; t=−3.579, p=0.0004 (***) | Offline learners score **2.93 points higher** on average — a statistically significant difference at the 0.001 level. |
| **Retention_Score** | Offline=74.72, Online=70.66; t=−3.648, p=0.0003 (***) | Offline learners retain material **4.06 points more** — again highly significant. |
| **Focus_Level** | Offline=71.80, Online=70.72; p=0.347 (ns) | The small 1.08-point gap in focus is **not statistically significant**; both groups exhibit similar focus distributions. |
| **Study_Hours** | Similar across modes; p>0.05 (ns) | Study time does not differ meaningfully between modes, ruling out hours studied as a confound for the performance gap. |

**Takeaway:** Offline learners outperform their online counterparts on *outcome* measures (exam score and retention), but there is **no significant difference in effort** (study hours) or momentary attention (focus level). This suggests the learning environment itself — not differential effort — may drive the performance gap.


## 3. Figure 1 — Score Distributions by Learning Mode

Side-by-side box plots for Exam Score, Retention Score, and Focus Level, coloured by Learning Mode. Significance brackets show results from the independent t-tests above.


In [ ]:
metrics  = ['Exam_Score', 'Retention_Score', 'Focus_Level']
ylabels  = ['Exam Score', 'Retention Score', 'Focus Level']

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Online vs. Offline Learning: Score Distributions', **FONT_TITLE, y=1.02)

for ax, metric, ylabel in zip(axes, metrics, ylabels):
    data_on  = online[metric].dropna()
    data_off = offline[metric].dropna()

    bp = ax.boxplot(
        [data_on, data_off],
        labels=['Online', 'Offline'],
        patch_artist=True,
        widths=0.5,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(linewidth=1.4),
        capprops=dict(linewidth=1.4),
        flierprops=dict(marker='o', markersize=4, alpha=0.5),
    )
    bp['boxes'][0].set_facecolor(PALETTE['Online'])
    bp['boxes'][1].set_facecolor(PALETTE['Offline'])
    bp['boxes'][0].set_alpha(0.75)
    bp['boxes'][1].set_alpha(0.75)

    ax.set_title(metric.replace('_', ' '), fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, **FONT_AXIS)
    ax.set_xlabel('Learning Mode', **FONT_AXIS)

    # Annotate medians
    for i, data in enumerate([data_on, data_off], 1):
        med = np.median(data)
        ax.text(i, med + 0.5, f'{med:.1f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold', color='black')

    # Significance bracket
    _, p = stats.ttest_ind(data_on, data_off)
    sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    y_top = max(data_on.max(), data_off.max()) + 2
    ax.annotate('', xy=(2, y_top), xytext=(1, y_top),
                arrowprops=dict(arrowstyle='-', lw=1.5))
    ax.text(1.5, y_top + 0.5, sig, ha='center', fontsize=11, fontweight='bold')

patches = [mpatches.Patch(color=PALETTE[m], alpha=0.75, label=m) for m in ['Online', 'Offline']]
fig.legend(handles=patches, loc='upper right', fontsize=10, title='Mode')
fig.tight_layout()

fig.savefig('fig1_score_distributions.pdf', dpi=150, bbox_inches='tight')
fig.savefig('fig1_score_distributions.png', dpi=150, bbox_inches='tight')
print('Saved fig1_score_distributions.pdf and fig1_score_distributions.png')
plt.show()

## 4. Figure 2 — Exam Score by Subject and Learning Mode

Grouped bar chart displaying mean Exam Score ± SEM for each subject, split by Learning Mode. The offline advantage is consistent across all five subjects.


In [ ]:
agg      = df.groupby(['Subject', 'Learning_Mode'])['Exam_Score'].agg(['mean', 'sem']).reset_index()
subjects = sorted(df['Subject'].unique())
x        = np.arange(len(subjects))
width    = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
for i, mode in enumerate(['Online', 'Offline']):
    sub = agg[agg['Learning_Mode'] == mode].set_index('Subject').reindex(subjects)
    ax.bar(x + i * width, sub['mean'], width,
           yerr=sub['sem'], capsize=4,
           color=PALETTE[mode], alpha=0.85, label=mode,
           error_kw=dict(elinewidth=1.5, ecolor='black'))
    for j, (m, s) in enumerate(zip(sub['mean'], sub['sem'])):
        ax.text(j + i * width, m + s + 0.8, f'{m:.1f}',
                ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x + width / 2)
ax.set_xticklabels(subjects, fontsize=11)
ax.set_xlabel('Subject', **FONT_AXIS)
ax.set_ylabel('Mean Exam Score', **FONT_AXIS)
ax.set_title('Mean Exam Score by Subject and Learning Mode\n(error bars = SEM)', **FONT_TITLE)
ax.legend(title='Learning Mode', fontsize=10)
ax.set_ylim(0, agg['mean'].max() + 12)
fig.tight_layout()

fig.savefig('fig2_exam_by_subject.pdf', dpi=150, bbox_inches='tight')
fig.savefig('fig2_exam_by_subject.png', dpi=150, bbox_inches='tight')
print('Saved fig2_exam_by_subject.pdf and fig2_exam_by_subject.png')
plt.show()

## 5. Correlation Analysis

We compute Pearson correlation coefficients between the main predictor variables (`Retention_Score`, `Focus_Level`, `Study_Hours`) and the outcome (`Exam_Score`), separately for each learning mode.


In [ ]:
corr_rows = []
for mode, grp in [('Online', online), ('Offline', offline)]:
    for col in ['Retention_Score', 'Focus_Level', 'Study_Hours']:
        r, p = stats.pearsonr(grp[col].dropna(), grp['Exam_Score'].dropna())
        sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        corr_rows.append({
            'Learning Mode': mode,
            'Predictor':     col,
            'r':             round(r, 4),
            'p-value':       round(p, 4),
            'Significance':  sig,
            'Strength':      'Very Strong' if abs(r) >= 0.7 else
                             ('Strong' if abs(r) >= 0.5 else
                             ('Moderate' if abs(r) >= 0.3 else 'Weak'))
        })

corr_df = pd.DataFrame(corr_rows).set_index(['Learning Mode', 'Predictor'])
print('Pearson Correlations with Exam_Score')
print('=' * 60)
corr_df

## 6. Figure 3 — Retention Score vs. Exam Score

Scatter plot with OLS regression lines per learning mode.
The near-identical slopes highlight that retention is an equally powerful predictor in both modes (r ≈ +0.80).


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for mode, grp in [('Online', online), ('Offline', offline)]:
    x_vals = grp['Retention_Score'].values
    y_vals = grp['Exam_Score'].values
    ax.scatter(x_vals, y_vals, c=PALETTE[mode], alpha=0.35,
               s=35, label=mode, edgecolors='none')
    slope, intercept, r, p, _ = stats.linregress(x_vals, y_vals)
    xr = np.linspace(x_vals.min(), x_vals.max(), 200)
    ax.plot(xr, intercept + slope * xr, color=PALETTE[mode],
            linewidth=2.2, label=f'{mode} fit  (r={r:+.3f}, p<0.001)')

ax.set_xlabel('Retention Score', **FONT_AXIS)
ax.set_ylabel('Exam Score', **FONT_AXIS)
ax.set_title('Retention Score vs. Exam Score by Learning Mode', **FONT_TITLE)
ax.legend(fontsize=10, title='Learning Mode / Fit')
fig.tight_layout()

fig.savefig('fig3_retention_vs_exam.pdf', dpi=150, bbox_inches='tight')
fig.savefig('fig3_retention_vs_exam.png', dpi=150, bbox_inches='tight')
print('Saved fig3_retention_vs_exam.pdf and fig3_retention_vs_exam.png')
plt.show()

## 7. Figure 4 — Focus Level vs. Exam Score

Scatter plot with OLS regression lines per learning mode.
Focus Level shows a moderate-to-strong correlation with Exam Score (r ≈ +0.48 Online, r ≈ +0.56 Offline).
The steeper slope for Offline learners suggests focus may confer a slightly larger benefit in face-to-face settings.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for mode, grp in [('Online', online), ('Offline', offline)]:
    x_vals = grp['Focus_Level'].values
    y_vals = grp['Exam_Score'].values
    ax.scatter(x_vals, y_vals, c=PALETTE[mode], alpha=0.35,
               s=35, label=mode, edgecolors='none')
    slope, intercept, r, p, _ = stats.linregress(x_vals, y_vals)
    xr = np.linspace(x_vals.min(), x_vals.max(), 200)
    ax.plot(xr, intercept + slope * xr, color=PALETTE[mode],
            linewidth=2.2, label=f'{mode} fit  (r={r:+.3f}, p<0.001)')

ax.set_xlabel('Focus Level', **FONT_AXIS)
ax.set_ylabel('Exam Score', **FONT_AXIS)
ax.set_title('Focus Level vs. Exam Score by Learning Mode', **FONT_TITLE)
ax.legend(fontsize=10, title='Learning Mode / Fit')
fig.tight_layout()

fig.savefig('fig4_focus_vs_exam.pdf', dpi=150, bbox_inches='tight')
fig.savefig('fig4_focus_vs_exam.png', dpi=150, bbox_inches='tight')
print('Saved fig4_focus_vs_exam.pdf and fig4_focus_vs_exam.png')
plt.show()

## 8. Figure 5 — Study Hours Distribution

KDE-smoothed histogram overlaid for each learning mode.
The near-identical distributions confirm that Online and Offline students invest similar amounts of study time,
making study hours an unlikely explanation for the observed performance gap.


In [ ]:
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(df['Study_Hours'].min() - 0.5,
                   df['Study_Hours'].max() + 0.5, 30)

for mode, grp in [('Online', online), ('Offline', offline)]:
    vals = grp['Study_Hours'].dropna().values
    ax.hist(vals, bins=bins, color=PALETTE[mode],
            alpha=0.45, label=f'{mode} (histogram)', density=True)
    kde = gaussian_kde(vals, bw_method=0.4)
    xr  = np.linspace(vals.min() - 0.3, vals.max() + 0.3, 300)
    ax.plot(xr, kde(xr), color=PALETTE[mode], linewidth=2.5,
            label=f'{mode} KDE')

ax.set_xlabel('Study Hours', **FONT_AXIS)
ax.set_ylabel('Density', **FONT_AXIS)
ax.set_title('Distribution of Study Hours by Learning Mode', **FONT_TITLE)
ax.legend(fontsize=10)
fig.tight_layout()

fig.savefig('fig5_study_hours_distribution.pdf', dpi=150, bbox_inches='tight')
fig.savefig('fig5_study_hours_distribution.png', dpi=150, bbox_inches='tight')
print('Saved fig5_study_hours_distribution.pdf and fig5_study_hours_distribution.png')
plt.show()

## 9. Figure 6 — Exam Score Heatmap: Subject × Learning Mode

A seaborn heatmap summarising mean Exam Score for every Subject × Learning Mode cell,
providing a compact overview of where the performance gap is largest.


In [ ]:
heat_data = df.pivot_table(values='Exam_Score',
                           index='Subject',
                           columns='Learning_Mode',
                           aggfunc='mean').round(2)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    heat_data,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    annot_kws={'size': 12, 'weight': 'bold'},
    ax=ax,
    cbar_kws={'label': 'Mean Exam Score', 'shrink': 0.85}
)
ax.set_title('Mean Exam Score: Subject × Learning Mode', **FONT_TITLE)
ax.set_xlabel('Learning Mode', **FONT_AXIS)
ax.set_ylabel('Subject', **FONT_AXIS)
ax.tick_params(axis='both', labelsize=11)
fig.tight_layout()

fig.savefig('fig6_subject_heatmap.pdf', dpi=150, bbox_inches='tight')
fig.savefig('fig6_subject_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved fig6_subject_heatmap.pdf and fig6_subject_heatmap.png')
plt.show()

# Also print the pivot table
heat_data['Diff (Offline - Online)'] = (heat_data['Offline'] - heat_data['Online']).round(2)
print('\nMean Exam Score pivot table:')
print(heat_data.to_string())

## 10. Summary of Findings

---

### Research Question 1 — Does learning mode significantly affect exam performance, retention, and focus?

**Yes**, but selectively:

| Outcome | Online | Offline | Δ (Off − On) | p-value | Verdict |
|---------|--------|---------|--------------|---------|--------|
| Exam Score | 70.21 | 73.14 | **+2.93** | 0.0004 (***) | Significant |
| Retention Score | 70.66 | 74.72 | **+4.06** | 0.0003 (***) | Significant |
| Focus Level | 70.72 | 71.80 | +1.08 | 0.347 (ns) | **Not significant** |
| Study Hours | similar | similar | ~0 | >0.05 (ns) | **Not significant** |

Offline learners score ~3 points higher on exams and retain material ~4 points better. Crucially, this gap cannot be attributed to differences in effort (study hours are equal) or momentary attention (focus levels are statistically identical). The advantage likely stems from structural features of the learning environment itself.

---

### Research Question 2 — What are the strongest predictors of Exam Score, and does learning mode moderate these relationships?

**Retention Score** is overwhelmingly the strongest predictor of Exam Score in both modes:

| Predictor | Online r | Offline r | Moderation? |
|-----------|----------|-----------|-------------|
| Retention_Score | **+0.795** | **+0.797** | Minimal — slopes nearly identical |
| Focus_Level | +0.483 | +0.563 | Moderate — offline focus has slightly more impact |
| Study_Hours | weak | weak | Negligible |

Learning mode **moderates the Focus Level → Exam Score** relationship to a small degree: higher focus translates into a marginally larger exam gain for offline students (r=+0.563 vs. +0.483). For retention, the relationship is essentially identical across modes — deep processing is equally beneficial regardless of delivery format.

---

### Research Question 3 — Do online and offline learners differ in subject-specific performance?

**Yes**, the offline advantage is present across **all five subjects** but varies in magnitude:

| Subject | Offline Mean | Online Mean | Gap (Off − On) |
|---------|-------------|-------------|----------------|
| History | highest gap | lowest | **+4.09** |
| English | | | +3.12 |
| Programming | | | +3.62 |
| Science | | | +2.69 |
| Math | narrowest gap | | **+1.49** |

History shows the largest offline premium (+4.09 pts), while Math shows the smallest (+1.49 pts). This pattern is consistent with the hypothesis that subjects requiring deeper contextual understanding and discussion (History, English) benefit more from face-to-face instruction, whereas structured problem-solving subjects (Math) are more resilient to delivery mode.

---

### Overall Conclusion

> Offline learning is associated with **statistically significant improvements** in exam performance and retention across all subjects. **Retention Score** is the strongest predictor of exam success (r ≈ +0.80) regardless of mode. The offline advantage is not explained by greater study time or focus, suggesting that the structural and social features of in-person instruction may be the key differentiator.
